In [1]:
import os
import shutil
import pickle
import boto3

### API Requirments for Deployment via Azure
- ```Dockerfile``` (for building environment)
- ```requirements.txt``` (x2; for installing correct versions of packages)
- ```cls_parser.pkl``` (for generating responses)
- ```app.py``` (for serving the Flask app)
- ```functions.py``` (helper functions for API)
- ```preprocessing.py``` (functions for shared preprocessing)
- ```api.py``` (for processing requests)

### Constants

In [2]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

Project: 20231010-gen-xii


### Create ```api``` directory

In [3]:
str_dirname = 'api'
try:
    os.mkdir(str_dirname)
except FileExistsError:
    pass

### Create ```web``` directory inside ```api``` directory

In [4]:
str_dirname = 'api/web'
try:
    os.mkdir(str_dirname)
except FileExistsError:
    pass

### Place files appropriately

### Write ```Dockerfile``` to ```api/web/``` directory

In [5]:
%%writefile api/web/Dockerfile

FROM python:3.9
    
WORKDIR usr/src/app

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# api.py
COPY api.py .

# functions.py
COPY functions.py .

# functions_counters.py
COPY functions_counters.py .

# functions_counters_pricing.py
COPY functions_counters_pricing.py .

# preprocessing.py
COPY preprocessing.py .

# cls_parser.pkl
COPY cls_parser.pkl .

# copy function code
COPY app.py .

# run script when image is run
CMD ["python", "-u", "app.py"]

Overwriting api/web/Dockerfile


### Load ```requirements.txt```

In [6]:
# root directory
str_filename = 'requirements.txt'
str_source = f'../01_flask_app/app/{str_filename}'
str_destination = f'./api/{str_filename}'
shutil.copyfile(str_source, str_destination)

'./api/requirements.txt'

In [7]:
# web directory
str_filename = 'requirements.txt'
str_source = f'../01_flask_app/app/{str_filename}'
str_destination = f'./api/web/{str_filename}'
shutil.copyfile(str_source, str_destination)

'./api/web/requirements.txt'

### Load other files

In [8]:
list_str_filenames = [
    'api.py',
    'app.py',
    'cls_parser.pkl',
    'functions.py',
    'functions_counters.py',
    'functions_counters_pricing.py',
    'preprocessing.py',
]
for str_filename in list_str_filenames:
    str_source = f'../01_flask_app/app/{str_filename}'
    str_destination = f'./api/web/{str_filename}'
    shutil.copyfile(str_source, str_destination)

### Write ```app.py``` to ```api/web/``` directory

In [9]:
%%writefile api/web/app.py

from flask import Flask, request, jsonify, Response
import pickle
import traceback
import logging
from waitress import serve
import pandas as pd 
pd.options.mode.chained_assignment = None # suppress warning

# set up logging
logging.basicConfig(
    filename='flask_app.log', 
    level=logging.DEBUG, 
    format='%(asctime)s %(levelname)s %(message)s',
)

# instantiate app
app = Flask(__name__)

# route the model to http://127.0.0.1:5000/
@app.route('/', methods=['GET','POST']) # GET for status code, POST for predictions
# logic for GET and POST requests
def predict():
    if request.method == 'GET':
        # log it
        logging.info('GET request received')
        # get status code
        int_status_code = Response(status=200).status_code
        # return
        return f'Status code: {int_status_code}'
    elif request.method == 'POST':
        try:
            # import parser
            str_message = 'Loading parser...'
            logging.info(str_message)
            print(str_message)
            print('')
            cls_parse_payload = pickle.load(open('cls_parser.pkl', 'rb'))
            
            # get payload
            str_message = 'Getting request...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_json_request = request.get_json()
            
            # parse payload
            str_message = 'Parsing payload...'
            logging.info(str_message)
            print(str_message)
            cls_parse_payload.get_data(dict_json_request)
            cls_parse_payload.engineer_pmt_hx()
            cls_parse_payload.shared_preprocessing()
            cls_parse_payload.generate_predictions()
            cls_parse_payload.apply_policies()
            cls_parse_payload.apply_chime()
            cls_parse_payload.adverse_action()
            cls_parse_payload.counter_offers()
            cls_parse_payload.generate_output()
            
            # extract output
            str_message = 'Extracting output...'
            logging.info(str_message)
            print(str_message)
            print('')
            dict_output = cls_parse_payload.dict_output
            # return output_final
            return dict_output['output_final']
        except Exception as e:
            str_message = 'Exception occurred'
            logging.error(
                str_message, 
                exc_info=True,
            )
            #return traceback in json
            return jsonify({'error': str(e)})

# run app
if __name__ == '__main__':
    # serve app
    serve(app, host='0.0.0.0', port=5000)

Overwriting api/web/app.py


### Build image

In [10]:
%%sh

# name the image
image=gen-xii-azure

# cd to web
cd ./api/web

# build image
docker build -t ${image} .

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 568B done
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.9
#2 DONE 0.3s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [ 1/11] FROM docker.io/library/python:3.9@sha256:5ea663a1c6ba266fdcac5949d1d2ea364ce30a2da92a3df95bb3c01437633ad9
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 3.48MB 0.0s done
#5 DONE 0.0s

#6 [10/11] COPY cls_parser.pkl .
#6 CACHED

#7 [ 9/11] COPY preprocessing.py .
#7 CACHED

#8 [ 2/11] WORKDIR usr/src/app
#8 CACHED

#9 [ 4/11] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [ 8/11] COPY functions_counters_pricing.py .
#10 CACHED

#11 [ 7/11] COPY functions_counters.py .
#11 CACHED

#12 [ 6/11] COPY functions.py .
#12 CACHED

#13 [ 5/11] COPY api.py .
#13 CACHED

#14 [ 3/11] COPY requirements.txt .
#14 CACHED

#15 [11/11] COP